# GHPA static-prior analysis (visualize + per-image eval + stratified comparison)

Runs the three analysis scripts on experiments trained with `colab_train.ipynb` (checkpoints on Google Drive):

1. `analysis/visualize_hpa.py` — captures the static gates `conv_*(BI(P))` of all six GHPA modules, plots them, and measures spatial non-uniformity and correlation with the dataset **lesion prior** (mean train mask), against an untrained model (P = ones).
2. `analysis/eval_per_image.py` — per-image DSC/IoU + lesion position/size descriptors (also with test-time shifts and post-hoc gate overrides on the learnable model).
3. `analysis/compare_runs.py` — compares runs/groups of seeds per lesion-position and lesion-size tertile, with bootstrap CIs and the **interaction test** (gain in central vs off-center images).

Everything is written next to each experiment under `<WORK_DIR>/analysis/`, i.e. persisted on Drive. A GPU is optional (the model is tiny); CPU works.

In [ ]:
# ---------------- Parameters ----------------
REPO_URL     = 'https://github.com/<your-username>/EGE-UNet.git'
DATASET      = 'isic17'                                          # 'isic17' | 'isic18'
DATA_ZIP     = '/content/drive/MyDrive/datasets/data_isic1718.zip'
RESULTS_ROOT = '/content/drive/MyDrive/EGE-UNet-results'

# experiments to analyse: label -> work_dir (must contain checkpoints/ and test_results.json)
RUNS = {
    'learnable_s42':   f'{RESULTS_ROOT}/egeunet_{DATASET}_learnable_s42',
    'frozen_ones_s42': f'{RESULTS_ROOT}/egeunet_{DATASET}_frozen_ones_s42',
    'none_s42':        f'{RESULTS_ROOT}/egeunet_{DATASET}_none_s42',
}
# groups of seeds for compare_runs (first group = reference). Add more seeds as they finish, e.g.
# 'learnable': ['learnable_s42', 'learnable_s43', 'learnable_s44']
GROUPS = {
    'learnable':   ['learnable_s42'],
    'frozen_ones': ['frozen_ones_s42'],
    'none':        ['none_s42'],
}
INPUT_SIZE     = 256
DEVICE         = 'cuda'                     # 'cpu' also fine
SHIFTS         = [(32, 0), (0, 32)]         # test-time translations (dx, dy) in pixels; [] to skip
GATE_OVERRIDES = ['spatial_mean', 'ones']   # post-hoc knock-outs on learnable runs; [] to skip

DATA_DIR = '/content/data_isic1718/' + ('isic2017' if DATASET == 'isic17' else 'isic2018')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir('/content/EGE-UNet'):
    !git clone {REPO_URL} /content/EGE-UNet
else:
    !git -C /content/EGE-UNet pull --ff-only
%cd /content/EGE-UNet
!pip install -q -r requirements.txt

In [ ]:
# Stage the dataset to local disk (see colab_train.ipynb for why)
if not os.path.isdir(DATA_DIR):
    !cp "{DATA_ZIP}" /content/data_isic1718.zip
    !unzip -q /content/data_isic1718.zip -d /content/data_isic1718
    !rm /content/data_isic1718.zip
!ls {DATA_DIR}

### 1+2. Visualize the gates and evaluate per image, for every run

Shifts and gate overrides are only meaningful on runs that have HPA gates (`learnable` / `frozen_ones`); `none` runs skip them automatically.

In [ ]:
import json

for label, wd in RUNS.items():
    print('=' * 100); print(label, wd)
    if not os.path.isdir(wd):
        print('  [skip] work_dir not found'); continue
    !python analysis/visualize_hpa.py --checkpoint "{wd}" --data-path "{DATA_DIR}" --dataset {DATASET} --input-size {INPUT_SIZE} --device {DEVICE}
    !python analysis/eval_per_image.py --checkpoint "{wd}" --data-path "{DATA_DIR}" --dataset {DATASET} --input-size {INPUT_SIZE} --device {DEVICE}
    has_gates = json.load(open(f'{wd}/analysis/hpa/hpa_stats.json')).get('hpa_mode') != 'none'
    for dx, dy in SHIFTS:
        !python analysis/eval_per_image.py --checkpoint "{wd}" --data-path "{DATA_DIR}" --dataset {DATASET} --input-size {INPUT_SIZE} --device {DEVICE} --shift {dx} {dy}
    if has_gates:
        for mode in GATE_OVERRIDES:
            !python analysis/eval_per_image.py --checkpoint "{wd}" --data-path "{DATA_DIR}" --dataset {DATASET} --input-size {INPUT_SIZE} --device {DEVICE} --gate-override {mode}

### Gate figures and statistics per run

In [ ]:
import pandas as pd
from IPython.display import Image, display, Markdown

for label, wd in RUNS.items():
    hpa = f'{wd}/analysis/hpa'
    if not os.path.isfile(f'{hpa}/hpa_stats.json'):
        continue
    display(Markdown(f'## {label}'))
    for fig in ('fig_summary_gxy_vs_prior.png', 'fig_summary_gxy_vs_prior_interior.png', 'fig_summary_stats.png'):
        if os.path.isfile(f'{hpa}/{fig}'):
            display(Image(filename=f'{hpa}/{fig}'))
    csv_path = f'{hpa}/hpa_stats_per_module.csv'
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        cols = ['module', 'which', 'gate', 'H', 'W', 'C', 'cv_interior_meanmap', 'r_prior_interior_meanmap',
                'r_radial_interior_meanmap', 'center_ratio_interior_meanmap', 'cv_interior_mean', 'p_mean_abs_dev_from_one_mean']
        display(df[df.gate == 'xy'][cols].round(4))
    # pooled numbers for the plain / shifted / overridden evaluations
    rows = []
    for f in sorted(os.listdir(f'{wd}/analysis')):
        if f.startswith('per_image_metrics') and f.endswith('.json'):
            js = json.load(open(f'{wd}/analysis/{f}'))
            rows.append({'eval': f[len('per_image_metrics'):-5] or '(plain)', **js['pooled'], 'mean_dsc': js['per_image']['mean_dsc']})
    if rows:
        display(pd.DataFrame(rows).round(4))

### 3. Stratified comparison between groups (reference = first group)

In [ ]:
group_args = ' '.join(f'--group {g}=' + ','.join(RUNS[r] for r in members if os.path.isdir(RUNS[r])) for g, members in GROUPS.items())
ref_dir = RUNS[GROUPS[list(GROUPS)[0]][0]]
!python analysis/compare_runs.py {group_args} --metric dsc --out-dir "{ref_dir}/analysis/compare"
display(Image(filename=f'{ref_dir}/analysis/compare/compare_runs_dsc.png'))
display(pd.read_csv(f'{ref_dir}/analysis/compare/compare_runs_dsc_interaction.csv').round(4))

# the same comparison on the shifted evaluations (robustness to translation)
for dx, dy in SHIFTS:
    print(f'\n=== shift ({dx}, {dy})')
    !python analysis/compare_runs.py {group_args} --metric dsc --csv-name per_image_metrics_shift{dx}_{dy}.csv --out-dir "{ref_dir}/analysis/compare_shift{dx}_{dy}"

### How to read the results

- **Is there a learned spatial prior at all?** `cv_interior` of the trained gates vs the untrained reference (which is exactly 0 in the interior — its only structure is the 1-px zero-padding border band). `p_mean_abs_dev_from_one` tells how far P moved from its init.
- **Is it a center bias?** `r_prior_interior` (correlation of the channel-mean |gate| with the lesion prior) and `r_radial_interior` (negative = emphasis at the center), `center_ratio_interior` > 1 = center emphasis. Look at `fig_summary_gxy_vs_prior_interior.png` — with the border removed, the interior structure is visible.
- **What does P contribute?** Pooled mIoU/DSC: `learnable` vs `frozen_ones` (spatial prior removed, per-channel scaling kept) vs `none` (no gating). The post-hoc `--gate-override spatial_mean` on the learnable checkpoint is the same question without retraining (but the rest of the network was co-adapted to the gate, so expect a larger drop than the retrained `frozen_ones`).
- **Does the gain depend on where the lesion is?** The interaction row of `compare_runs`: `diff_T1 - diff_T3` (central minus off-center) with a bootstrap CI that excludes 0 means the advantage of the reference over that group is concentrated in centered lesions — evidence that P is a positional prior. Shifted evaluations (`--shift`) test the same thing by moving lesions away from where they were during training.
- Differences of a few tenths of a mIoU point are within the seed noise reported in the paper (std ≈ 0.10); use the `GROUPS` mechanism with ≥ 3 seeds before drawing conclusions.